# Lab 1 — Entity Preparation

In this lab, we will:

- Define a *watch list* of entities we care about (names, aliases, types, and context)
- Validate inputs and dependencies so later labs fail fast and predictably
- Run the entity preparation pipeline on a small dataset and inspect the outputs

This lab is:
- ✅ Standalone (does not rely on `pipeline_state` from other labs)
- ❌ Not a production pipeline
- ✅ Designed for learning through inspection

You should come away understanding:
- What *entity context* is and why it matters for resolution quality
- How metadata (aliases, type, priority) affects matching downstream
- Where enrichment fits in, and how to control it


In [ ]:
# Lab output controls
VERBOSE = False  # set True if you want extra explanatory output

def vprint(*args, **kwargs):
    if VERBOSE:
        vprint(*args, **kwargs)

## Lab Step 1 — Setup and Imports

This step exists to standardize output (so you can focus on learning) and to ensure imports work before we touch data or Elasticsearch.

**What you’ll do:** Import dependencies and load config.


**Note:** If this step fails, fix it before continuing. Most later steps depend on these basics.

**Checkpoint:** You should see confirmation messages with no errors.


In [ ]:
# Setup and Imports
import sys
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
import warnings
import logging

# Configure logging FIRST - before any imports that might set up loggers
# Only show WARNING and ERROR messages to avoid confusing red boxes
logging.basicConfig(level=logging.WARNING, force=True)

# Set all known loggers to WARNING level BEFORE importing modules
loggers_to_suppress = [
    "entity_resolution_demo",
    "entity_resolution_demo.entity_preparation",
    "entity_resolution_demo.entity_preparation.entity_enricher",
    "entity_resolution_demo.entity_preparation.entity_indexer", 
    "entity_resolution_demo.entity_preparation.entity_watch_list",
    "entity_resolution_demo.search",
    "entity_resolution_demo.search.elastic_client",
    "entity_resolution_demo.pipeline_runner",
    "entity_resolution_demo.pipeline_runner.utils",
    "elastic_transport",
    "elastic_transport.transport",
    "elasticsearch",
    "urllib3",
    "urllib3.connectionpool",
    "requests",
    "requests.packages.urllib3",
    "httpx",
    "httpcore"
]

for logger_name in loggers_to_suppress:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Also suppress any logger that starts with these prefixes
for logger_name in ["entity_resolution_demo", "elastic", "urllib3"]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

warnings.filterwarnings('ignore')
vprint("ℹ️  Logging configured to show only warnings and errors (INFO messages suppressed)")

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import project modules
from entity_resolution_demo.pipeline_runner.config import load_config
from entity_resolution_demo.search.elastic_client import ElasticClient
from entity_resolution_demo.entity_preparation.entity_watch_list import EntityWatchList
from entity_resolution_demo.entity_preparation.entity_enricher import EntityEnricher
from entity_resolution_demo.entity_preparation.entity_indexer import EntityIndexer
# from entity_resolution_demo.entity_preparation.entity_preparation import run_entity_preparation
from entity_resolution_demo.entity_preparation.entity_enricher import EnrichedEntity
from entity_resolution_demo.entity_preparation.entity_preparation import run_entity_preparation

# Load configuration
config = load_config()
print("✅ Configuration loaded")


# Force override any logger levels that might have been set during import
# This is a more aggressive approach to ensure INFO messages are suppressed
for logger_name in loggers_to_suppress:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.WARNING)
    # Also disable propagation to parent loggers
    logger.propagate = False

print("✅ All imports and initialization successful")

## Lab Step 2 — Validate Required Data Files

This step exists to fail early if required inputs are missing. Later steps assume these files exist and are valid.

**What you’ll do:** Confirm the required local data files are present.


**Note:** If this step fails, fix it before continuing. Most later steps depend on these basics.

**Checkpoint:** You should see checks pass for expected files.


In [ ]:
# Check required data files exist
# Note: config was already loaded in the previous cell, but we reload it here
# for clarity in this standalone section
minimal_entities_path = Path("minimal_entities.json")
if not minimal_entities_path.exists():
    raise FileNotFoundError(f"❌ Minimal entities file not found: {minimal_entities_path}")

print("✅ Required data files found")

# Load minimal entities dataset
with open(minimal_entities_path, 'r') as f:
    entities_data = json.load(f)

print(f"✅ Loaded minimal entities dataset: {len(entities_data.get('entities', []))} entities")

# Show sample entities
vprint("\n📋 Sample entities from minimal dataset:")
for i, entity in enumerate(entities_data.get('entities', [])[:3]):
    name = entity.get('name', 'Unknown')
    entity_type = entity.get('entity_type', 'PERSON')
    aliases = entity.get('aliases', [])
    vprint(f"   {i+1}. {name} ({entity_type}) - {len(aliases)} aliases")
    if aliases:
        vprint(f"      Aliases: {', '.join(aliases[:2])}{'...' if len(aliases) > 2 else ''}")

## Lab Step 3 — Validate Elasticsearch Connection

This step exists to verify we can talk to Elasticsearch *now*, before we do any indexing. If this fails, stop here—everything downstream depends on it.

**What you’ll do:** Create an Elasticsearch client and verify connectivity.


**Checkpoint:** You should see a successful connection/ping or basic cluster info.


In [ ]:
# Verify Elasticsearch connection
# Note: elastic_client was already initialized in the setup cell,
# but we verify the connection here to ensure it's working
# Initialize Elasticsearch client (if not already initialized)
try:
    elastic_client
except NameError:
    elastic_client = ElasticClient(config, allow_local_fallback=False)
    print("✅ Elasticsearch client initialized")

try:
    if elastic_client.check_connection():
        print("✅ Elasticsearch connection successful")
    else:
        raise ConnectionError("Failed to connect to Elasticsearch")
except Exception as e:
    print(f"❌ Elasticsearch connection failed: {e}")
    raise

## Lab Step 4 — Initialize Entity Preparation Components

This step exists to initialize the core components we’ll reuse throughout the lab series. Keeping this wiring consistent makes later debugging much easier.

**What you’ll do:** Instantiate the pipeline components used throughout the lab.


**Checkpoint:** You should see component initialization messages.


In [ ]:
# Initialize Entity Preparation Components
vprint("🔧 Initializing entity preparation components...")

# 1. EntityWatchList
watch_list = EntityWatchList()
print("✅ EntityWatchList initialized")

# 2. EntityEnricher
entity_enricher = EntityEnricher()
print("✅ EntityEnricher initialized")

# 3. EntityIndexer
entity_indexer = EntityIndexer(elastic_client=elastic_client, config=config)
print("✅ EntityIndexer initialized")

vprint("\n🎯 All components ready for entity preparation!")

## Lab Step 5 — Load and Validate Entities

This step exists to load a small, known-good dataset and validate its shape. We want deterministic inputs before enrichment or indexing.

**What you’ll do:** Load entities into the watch list and validate structure.


**Checkpoint:** You should see entity counts and a sample entity.


In [ ]:
# 🎯 Lab Step 5 — Load and validate the entity dataset
# Why this step exists:
#   This step converts the raw entity dataset into a canonical EntityWatchList.
#   Everything downstream (enrichment, indexing, matching) depends on this structure.
#   We intentionally fail early if the dataset shape is unexpected.

print("🎯 Lab Step 5: Load and validate the entity dataset")
print("=" * 60)

from entity_resolution_demo.entity_preparation.entity_watch_list import EntityWatchList

# ------------------------------------------------------------------
# Sanity check: ensure entity dataset was loaded earlier
# ------------------------------------------------------------------
if "entities_data" not in globals():
    raise RuntimeError(
        "entities_data is not defined. "
        "Ensure the entity dataset is loaded in the earlier data-loading step."
    )

raw = entities_data

# ------------------------------------------------------------------
# Normalize dataset shape into a list of entity dicts
# ------------------------------------------------------------------
if isinstance(raw, dict):
    # Common shape: {"entities": [ {...}, {...} ]}
    if "entities" in raw and isinstance(raw["entities"], list):
        entities_list = raw["entities"]
    else:
        raise TypeError(
            "entities_data is a dict but does not contain an 'entities' list. "
            f"Available keys: {list(raw.keys())[:20]}"
        )
elif isinstance(raw, list):
    entities_list = raw
else:
    raise TypeError(
        f"entities_data must be a dict or list, got: {type(raw)}"
    )

if not entities_list:
    raise ValueError("Entity dataset is empty after normalization.")

if not isinstance(entities_list[0], dict):
    raise TypeError(
        "Entity entries must be dictionaries with keys like 'name' and 'aliases'. "
        f"Found first entry type: {type(entities_list[0])}, "
        f"value: {repr(entities_list[0])[:200]}"
    )

print(f"✅ Loaded entity records: {len(entities_list)}")

# ------------------------------------------------------------------
# Build watch list
# ------------------------------------------------------------------
watch_list = EntityWatchList()

for ent in entities_list:
    if "name" not in ent or not ent["name"]:
        raise ValueError(f"Entity missing required 'name' field: {ent}")

    watch_list.add_entity(
        name=ent["name"],
        entity_type=ent.get("entity_type", "PERSON"),
        aliases=ent.get("aliases", []),
        metadata={
            "description": ent.get("description", ""),
            "priority": ent.get("priority", "medium"),
        },
    )

# ------------------------------------------------------------------
# Always-visible lab sanity checks
# ------------------------------------------------------------------
total_entities = len(watch_list.get_all_entities())
with_aliases = sum(1 for e in watch_list.get_all_entities() if e.aliases)

print(f"✅ Watch list created with {total_entities} entities")
print(f"   Entities with aliases: {with_aliases}")

# Show exactly one example (lab-style, not noisy)
example = watch_list.get_all_entities()[0]
print("\n🔍 Example entity:")
print(f"   Name: {example.name}")
print(f"   Type: {example.entity_type}")
print(f"   Aliases: {example.aliases}")
print(f"   Metadata keys: {list((example.metadata or {}).keys())}")

print("\n✅ Entity dataset successfully validated and loaded into watch list.")


## Lab Step 6 — Generate and Manage Entity Metadata

This step exists to make entity metadata explicit (aliases, types, priority). These choices have outsized impact on matching behavior later.

**What you’ll do:** Derive metadata used for downstream enrichment and indexing.


**Note:** Later steps assume this completes successfully (indexing + state). If it fails, you can still read the notebook, but execution will block.

**Checkpoint:** You should see metadata artifacts created/updated.


In [ ]:
# Metadata Management — Make entity metadata explicit (Lab-Friendly Output)

print("📋 Lab Step 6: Inspect and validate entity metadata")
print("=" * 60)

total = len(watch_list.entities)
priorities = {"high": 0, "medium": 0, "low": 0}
for e in watch_list.entities.values():
    p = (e.priority or "medium").lower()
    priorities[p] = priorities.get(p, 0) + 1

# Always-visible summary
print(f"✅ Metadata summary")
print(f"   Total entities: {total}")
print(f"   Priority: " + ", ".join([f"{k}={v}" for k, v in priorities.items()]))

# Show one example of “what metadata looks like”
example = next(iter(watch_list.entities.values())) if watch_list.entities else None
if example:
    print("   Example metadata snapshot:")
    print(f"   - {example.name}")
    print(f"     priority={example.priority} | aliases={len(example.aliases or [])}")
    # description can be long; show a short preview
    desc = (example.description or "").strip().replace("\n", " ")
    if desc:
        print(f"     description preview: {desc[:120]}{'...' if len(desc) > 120 else ''}")

# Optional detail
vprint("\n🏷️ Entity type distribution (VERBOSE):")
type_counts = {}
for e in watch_list.entities.values():
    type_counts[e.entity_type] = type_counts.get(e.entity_type, 0) + 1
for t, c in sorted(type_counts.items(), key=lambda x: (-x[1], x[0])):
    vprint(f"   {t}: {c}")



## Lab Step 7 — Enrich Entities with Wikipedia Context

This step exists to demonstrate enrichment and how context is selected. Guardrail: enrichment is optional—later labs may disable external sources for reproducibility.

**What you’ll do:** Add contextual text for improved matching and searchability.


**Checkpoint:** You should see enriched fields for at least one entity.


In [ ]:
# Wikipedia Context Enrichment — Demonstrate enrichment (with a concrete example)

print("🌐 Lab Step 7: Enrich one entity with external context (Wikipedia)")
print("=" * 60)

# NOTE: This step is intentionally a *demonstration*.
# Later labs (and some evaluation modes) may disable external sources for reproducibility.

# Pick a single entity to keep output compact and deterministic.
entity = next(iter(watch_list.entities.values())) if watch_list.entities else None
if not entity:
    raise ValueError("No entities available in watch_list. Run Step 5 first.")

print(f"🔍 Enriching example entity: {entity.name}")

try:
    enriched_entity = entity_enricher.enrich_entity(
        name=entity.name,
        source_context=entity.description,
        aliases=entity.aliases
    )

    ctx = (enriched_entity.entity_context or "").strip().replace("\n", " ")
    preview = ctx[:200] + ("..." if len(ctx) > 200 else "")

    print("✅ Enrichment complete")
    print(f"   Context length: {len(ctx)} characters")
    print(f"   Confidence: {getattr(enriched_entity, 'confidence_score', None)}")
    print(f"   Source: {getattr(enriched_entity, 'enrichment_source', 'unknown')}")
    print(f"   Context preview: {preview}")

    # Optional deep-dive
    if getattr(enriched_entity, "alternative_contexts", None):
        vprint(f"\n🔄 Alternative contexts available: {len(enriched_entity.alternative_contexts)}")
        for i, alt in enumerate(enriched_entity.alternative_contexts[:2], 1):
            alt = (alt or "").strip().replace("\n", " ")
            vprint(f"   {i}. {alt[:160]}{'...' if len(alt) > 160 else ''}")

except Exception as e:
    print(f"❌ Enrichment failed: {e}")
    print("   This lab can still continue, but you may want to check Wikipedia/network access.")


## Lab Step 8 — Create the Entity Index and Populate It

This step exists to make entities searchable by indexing them into Elasticsearch. Guardrail: indexing is a prerequisite for retrieval-based matching.

**What you’ll do:** Create the index mappings and index prepared entities.


**Note:** Later steps assume this completes successfully (indexing + state). If it fails, you can still read the notebook, but execution will block.

**Checkpoint:** You should see index creation and a non-zero indexed count.


In [ ]:
# Index Creation and Entity Population (Lab-Friendly Output)

print("🔧 Lab Step 8: Create the entity index and verify population")
print("=" * 60)

# Create index with semantic mappings (real implementation)
try:
    print("🔧 Creating Elasticsearch index with semantic mappings...")
    success = entity_indexer.create_indices()
    if not success:
        raise RuntimeError("EntityIndexer.create_indices() returned False")

    index_name = getattr(entity_indexer, "entity_index", None) or getattr(entity_indexer, "index_name", None)
    if not index_name:
        raise AttributeError("Could not determine index name from EntityIndexer (expected .entity_index or .index_name).")

    print(f"✅ Index ready: {index_name}")

    # Index a small sample to keep this lab step fast and inspectable
    sample_entities = list(watch_list.entities.values())[:3]
    print(f"📚 Indexing a small sample for verification: {len(sample_entities)} entities")

    for i, entity in enumerate(sample_entities, 1):
        enriched_entity = entity_enricher.enrich_entity(
            name=entity.name,
            source_context=entity.description,
            aliases=entity.aliases
        )

        # Add fields EntityIndexer expects (kept from prior version)
        enriched_entity.entity_type = entity.entity_type
        enriched_entity.description = entity.description

        entity_indexer.index_entity(enriched_entity)
        vprint(f"   {i}. Indexed: {entity.name}")

    # Refresh and verify
    try:
        elastic_client.es.indices.refresh(index=index_name)
    except Exception:
        pass

    count_response = elastic_client.es.count(index=index_name)
    doc_count = count_response.get("count", 0)

    print(f"✅ Index now contains {doc_count} documents (after refresh)")
    if doc_count < len(sample_entities):
        print("   ℹ️ Note: document count is lower than the number we attempted to index.")
        print("      This can happen if indexing failed silently or the client is pointing at a different cluster.")
        print("      Check Elasticsearch logs / client config if this persists.")

    # Optional mapping preview
    try:
        mapping = elastic_client.es.indices.get_mapping(index=index_name)
        props = mapping[index_name]["mappings"]["properties"]
        semantic_fields = [k for k, v in props.items() if isinstance(v, dict) and v.get("type") == "semantic_text"]
        vprint("\n🔍 Mapping preview (VERBOSE):")
        vprint(f"   Total fields: {len(props)}")
        vprint(f"   Semantic fields: {semantic_fields}")
    except Exception as e:
        vprint(f"⚠️ Could not fetch mapping (VERBOSE): {e}")

except Exception as e:
    print(f"❌ Step 8 failed: {e}")
    raise


## Lab Step 9 — Process and Inspect a Single Entity

This step exists to inspect one entity end-to-end so you understand the intermediate objects and fields before we run batch processing.

**What you’ll do:** Run preparation on one entity and inspect the final document.


**Checkpoint:** You should see the enriched entity and the stored document.


In [ ]:
# Lab Step 9 — End-to-End Entity Processing (Single Example)
# Goal: show ONE concrete entity flowing through enrichment → indexing → retrieval checks.
# This is a lab checkpoint, so output is intentionally small but inspectable.

from pprint import pformat

print("🧪 Lab Step 9 — End-to-End Entity Processing (Single Example)")
print("=" * 60)

# --- Preconditions ---
if "watch_list" not in globals():
    raise RuntimeError("watch_list not found. Run the earlier watch list step first.")
if "entity_enricher" not in globals():
    raise RuntimeError("entity_enricher not found. Run Step 6 (enrichment/indexing init) first.")
if "entity_indexer" not in globals():
    raise RuntimeError("entity_indexer not found. Run Step 6 (enrichment/indexing init) first.")
if "elastic_client" not in globals():
    raise RuntimeError("elastic_client not found. Run Step 2 first.")

entities = list(watch_list.get_all_entities())
if not entities:
    raise ValueError("watch_list contains no entities.")

# Choose a deterministic example entity (first in list)
e = entities[0]
name = getattr(e, "name", None)
aliases = list(getattr(e, "aliases", []) or [])
metadata = dict(getattr(e, "metadata", {}) or {})
explicit_context = (metadata.get("explicit_context") or metadata.get("description") or "").strip()

print("Selected entity:")
print(f"  - Name: {name}")
print(f"  - Aliases (sample): {aliases[:5] if aliases else '—'}")
if explicit_context:
    preview = explicit_context[:160].replace("\n", " ")
    print(f"  - Context preview: {preview}{'…' if len(explicit_context) > 160 else ''}")
else:
    print("  - Context preview: —")

# --- Enrich (context-only) ---
print("\nEnrichment result:")
enriched = entity_enricher.enrich_entity(
    name=name,
    source_context=explicit_context if explicit_context else None,
    aliases=aliases if aliases else None,
)

# Ensure dataset context wins (if this object supports it)
if explicit_context and hasattr(enriched, "entity_context"):
    enriched.entity_context = explicit_context

ctx = getattr(enriched, "entity_context", None) or explicit_context
ctx_len = len(ctx) if isinstance(ctx, str) else 0
print(f"  - Context source: dataset-provided")
print(f"  - Context length: {ctx_len} chars")

# --- Index the single entity ---
index_name = getattr(watch_list, "index_name", None) or getattr(entity_indexer, "entity_index", None) or getattr(entity_indexer, "index_name", None)
if not index_name:
    raise RuntimeError("Could not determine entity index name from watch_list or entity_indexer.")

print("\nIndexing result:")
ok = entity_indexer.index_entity(enriched)
print(f"  - Indexed into: {index_name}")
print(f"  - Index call returned: {'✅ success' if ok else '⚠️ reported failure'}")

# Refresh + count (source of truth)
try:
    elastic_client.es.indices.refresh(index=index_name)
except Exception:
    pass

try:
    count = elastic_client.es.count(index=index_name).get("count", None)
except Exception as ex:
    count = None
    vprint(f"Count failed: {ex}")

print(f"  - Total documents in index (post-refresh): {count if count is not None else 'unknown'}")

# --- Verification checks (retrieve by name and alias) ---
print("\nVerification checks:")
def _first_hit_total(resp: dict) -> int:
    # supports ES 7/8 total formats
    total = resp.get("hits", {}).get("total", 0)
    if isinstance(total, dict):
        return int(total.get("value", 0))
    return int(total)

def _search_entity(term: str) -> int:
    # Be robust to mapping differences (name, name.keyword, entity_name, aliases, aliases.keyword)
    should = []
    for field in ["name", "name.keyword", "entity_name", "entity_name.keyword", "aliases", "aliases.keyword"]:
        should.append({"term": {field: term}})
        # also try match for text fields
        should.append({"match": {field: {"query": term, "operator": "and"}}})

    resp = elastic_client.es.search(
        index=index_name,
        size=3,
        query={"bool": {"should": should, "minimum_should_match": 1}},
        _source=["name", "entity_name", "aliases"],
    )
    return _first_hit_total(resp)

# canonical name retrieval
name_hits = _search_entity(name)
print(f"  - Retrieved by canonical name: {'✅' if name_hits > 0 else '❌'} (hits={name_hits})")

# alias retrieval (one example alias if available)
if aliases:
    alias_term = aliases[0]
    alias_hits = _search_entity(alias_term)
    print(f"  - Retrieved by alias '{alias_term}': {'✅' if alias_hits > 0 else '❌'} (hits={alias_hits})")
else:
    print("  - Retrieved by alias: — (no aliases on this entity)")

vprint("\nDebug (verbose): enriched entity object:")
vprint(pformat(getattr(enriched, "model_dump", lambda: enriched)(), width=100) if hasattr(enriched, "model_dump") else pformat(enriched, width=100))

print("\n✅ Step 9 complete: one entity successfully flowed through enrichment → indexing → retrieval checks.")


## Lab Step 10 — Run the Full Pipeline (Batch Processing)

This step exists to run the same pipeline at batch scale and produce artifacts we can reuse or analyze.

**What you’ll do:** Run the full preparation pipeline across the dataset.


**Note:** Later steps assume this completes successfully (indexing + state). If it fails, you can still read the notebook, but execution will block.

**Checkpoint:** You should see progress and a completion summary.


In [ ]:
# 🔄 Complete Entity Preparation Pipeline - Batch Processing
vprint("🔄 Complete Entity Preparation Pipeline - Batch Processing")
vprint("=" * 60)

# This cell runs the complete entity preparation pipeline
# It will process all entities, enrich them, and save the state
vprint("📋 Running complete entity preparation pipeline...")
vprint("   - Processing all entities from minimal dataset")
vprint("   - Enriching with Wikipedia context (including multilingual support)")
vprint("   - Indexing in Elasticsearch with semantic mappings")
vprint("   - Saving complete state to pipeline_state/entity_preparation_state.json")
vprint("\n💡 Note: If you see warnings about state files, this is expected on the first run.")
vprint("   The pipeline will create a new state file after processing completes.")

try:
    # Run the complete entity preparation pipeline
    vprint(f"\n🔧 Starting entity preparation pipeline...")
    
    # Use the actual run_entity_preparation function from the pipeline
    vprint(f"\n🔧 Starting entity preparation pipeline...")
    
    # Set up the data file path in config
    if 'entity_preparation' not in config:
        config['entity_preparation'] = {}
    config['entity_preparation']['data_file'] = 'minimal_entities.json'
    
    # Run the actual entity preparation pipeline
    pipeline_success, pipeline_state = run_entity_preparation(
        config=config,
        state_dir="pipeline_state",
        verify=True
    )
    
    if pipeline_success:
        print(f"✅ Pipeline execution completed!")
        print(f"   - Entities processed (state): {len(pipeline_state.get('enriched_entities', []))}")
        print(f"   - Entities indexed (state): {len(pipeline_state.get('indexed_entities', []))}")
        print(f"   - Index name: {pipeline_state.get('entity_index_name', 'Unknown')}")
        # Validate the index contents directly in Elasticsearch (source of truth)
        index_name = pipeline_state.get('entity_index_name') or getattr(entity_indexer, 'entity_index', None)
        if index_name:
            try:
                elastic_client.es.indices.refresh(index=index_name)
                doc_count = elastic_client.es.count(index=index_name).get('count', 0)
                print(f"   - Elasticsearch documents in index: {doc_count}")
            except Exception as e:
                vprint(f"   (Could not count docs in ES: {e})")
        vprint(f"   - State saved to: pipeline_state/entity_preparation_state.json")
        
        # Show sample entities from the state
        enriched_entities = pipeline_state.get('enriched_entities', [])
        if enriched_entities:
            vprint(f"\n🔍 Sample Enriched Entities:")
            for i, entity in enumerate(enriched_entities[:3]):
                name = entity.get('name', 'Unknown')
                context_length = len(entity.get('entity_context', ''))
                confidence = entity.get('confidence_score', 0.0)
                source = entity.get('enrichment_source', 'Unknown')
                
                vprint(f"   {i+1}. {name}")
                vprint(f"      Context: {context_length} chars (confidence: {confidence:.3f})")
                vprint(f"      Source: {source}")
        
        print(f"\n✅ Complete pipeline execution successful!")
        vprint(f"   - All entities have been processed and enriched")
        vprint(f"   - State file has been updated with latest results")
        vprint(f"   - Ready for downstream processing (article processing, entity matching)")
    else:
        print(f"❌ Pipeline execution failed!")
        vprint(f"   - Check the error messages above for details")
        vprint(f"   - Verify Elasticsearch and Wikipedia API connections")
    
except Exception as e:
    print(f"❌ Error during pipeline execution: {e}")
    vprint(f"   This might indicate a configuration issue")
    vprint(f"   Check Elasticsearch and Wikipedia API connections")
    vprint(f"   The pipeline includes error handling and fallback strategies")

## Lab Step 11 — Review Saved Pipeline State

This step exists to show what we persist for reproducibility and why. If you can’t reproduce a run, you can’t debug it.

**What you’ll do:** Inspect the saved outputs/state produced by the pipeline.


**Checkpoint:** You should see paths/files and a brief summary of contents.


In [ ]:
# Pipeline State Demonstration
vprint("📁 Pipeline State: Saved Results")
vprint("=" * 50)

# Check if state file exists
# Note: The state file is saved in pipeline_state/ (relative to notebook directory)
state_file = Path("pipeline_state/entity_preparation_state.json")
if state_file.exists():
    print(f"✅ Pipeline state file found: {state_file}")
    
    # Load and analyze the state
    with open(state_file, 'r') as f:
        state_data = json.load(f)
    
    vprint(f"\n📊 State File Analysis:")
    vprint(f"   Total entities processed: {len(state_data.get('enriched_entities', []))}")
    vprint(f"   Processing timestamp: {state_data.get('timestamp', 'Unknown')}")
    vprint(f"   Pipeline version: {state_data.get('pipeline_version', 'Unknown')}")
    
    # Show sample enriched entities
    enriched_entities = state_data.get('enriched_entities', [])
    if enriched_entities:
        vprint(f"\n🔍 Sample Enriched Entities:")
        for i, entity in enumerate(enriched_entities[:3]):
            name = entity.get('name', 'Unknown')
            entity_type = entity.get('entity_type', 'Unknown')
            aliases = entity.get('aliases', [])
            context_length = len(entity.get('entity_context', ''))
            confidence = entity.get('confidence_score', 0.0)
            
            vprint(f"   {i+1}. {name} ({entity_type})")
            vprint(f"      Aliases: {len(aliases)} - {aliases[:2] if aliases else 'None'}")
            vprint(f"      Context: {context_length} chars (confidence: {confidence:.3f})")
    
    # Show processing statistics
    stats = state_data.get('processing_stats', {})
    if stats:
        vprint(f"\n📈 Processing Statistics:")
        print(f"   Entities enriched (state): {stats.get('entities_enriched', 0)}")
        print(f"   Entities indexed (state): {stats.get('entities_indexed', 0)}")
        vprint(f"   Enrichment errors: {stats.get('enrichment_errors', 0)}")
        vprint(f"   Indexing errors: {stats.get('indexing_errors', 0)}")
    
    # Show index information
    index_info = state_data.get('index_info', {})
    if index_info:
        vprint(f"\n🔍 Index Information:")
        vprint(f"   Index name: {index_info.get('index_name', 'Unknown')}")
        vprint(f"   Document count: {index_info.get('document_count', 0)}")
        vprint(f"   Index status: {index_info.get('status', 'Unknown')}")
    
    print(f"\n✅ Pipeline state analysis complete!")
    vprint(f"   - Shows the complete state of the entity preparation pipeline")
    vprint(f"   - Demonstrates what information is preserved for downstream processing")
    vprint(f"   - Provides insight into pipeline continuity and state management")
    
else:
    print(f"❌ Pipeline state file not found: {state_file}")
    vprint(f"   This indicates the pipeline hasn't been run yet")
    vprint(f"   Run the complete pipeline to generate the state file")

## Lab Step 12 — Dataset Overview for Scenarios

This step exists to orient you to the scenario dataset used in later labs and blogs.

**What you’ll do:** Review the scenario dataset used in the hands-on exercises.


**Checkpoint:** You should see dataset size and representative examples.


In [ ]:
# Dataset Overview for Educational Scenarios
print("📊 Dataset Overview for Educational Scenarios")
print("=" * 60)

# Analyze our minimal entities dataset
print("📋 Minimal Entities Dataset:")
print(f"   Total entities: {len(entities_data.get('entities', []))}")

# Show a small, always-visible preview
sample_entities = entities_data.get('entities', [])[:3]
if sample_entities:
    print("   Sample entities:")
    for e in sample_entities:
        name = e.get('name') if isinstance(e, dict) else str(e)
        print(f"   - {name}")

# Show sample entities with their characteristics
vprint(f"\n🎯 Sample Entities:")
for i, entity in enumerate(entities_data.get('entities', [])[:5]):
    name = entity.get('name', 'Unknown')
    entity_type = entity.get('entity_type', 'PERSON')
    priority = entity.get('priority', 'medium')
    aliases = entity.get('aliases', [])
    description = entity.get('description', 'No description')[:50]
    
    vprint(f"   {i+1}. {name} ({entity_type})")
    vprint(f"      Priority: {priority}")
    vprint(f"      Aliases: {len(aliases)} - {aliases[:2] if aliases else 'None'}")
    vprint(f"      Description: {description}...")

# Analyze entity characteristics
vprint(f"\n📊 Entity Characteristics:")
entity_types = {}
priorities = {}
alias_counts = []

for entity in entities_data.get('entities', []):
    # Entity types
    entity_type = entity.get('entity_type', 'PERSON')
    entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    # Priorities
    priority = entity.get('priority', 'medium')
    priorities[priority] = priorities.get(priority, 0) + 1
    
    # Alias counts
    aliases = entity.get('aliases', [])
    alias_counts.append(len(aliases))

vprint(f"   Entity types: {entity_types}")
vprint(f"   Priorities: {priorities}")
vprint(f"   Average aliases per entity: {sum(alias_counts)/len(alias_counts):.1f}")
vprint(f"   Entities with aliases: {sum(1 for count in alias_counts if count > 0)}")

print(f"\n✅ Dataset overview complete!")
vprint(f"   - Shows the foundation for our educational scenarios")
vprint(f"   - Demonstrates the diversity of entities and metadata")
vprint(f"   - Provides context for understanding preparation challenges")

## Lab Step 13 — Scenario 1 — Explicit Context Processing

This step exists to show the “happy path” where entities come with explicit context (no enrichment needed).

**What you’ll do:** See how explicit context changes resolution behavior.


**Checkpoint:** You should observe differences in prepared output.


In [ ]:
# Scenario 1: Explicit Context Processing
print("🎯 Scenario 1: Explicit Context Processing")
print("=" * 50)

# Demonstrate how the system handles entities with explicit context
# This shows real implementation of context-aware processing
vprint("\n 📋 Explicit Context Processing Demonstration:")
vprint("   - Entities with explicit context get priority processing")
vprint("   - Explicit context is more reliable than external sources")
vprint("   - Saves API calls and improves processing speed")
vprint("   - Provides consistent and controlled context quality")

# Find entities with explicit context from the dataset
entities_with_explicit_context = []
for entity in entities_data.get('entities', []):
    # Look for entities with explicit_context field
    explicit_context = entity.get('explicit_context', '')
    if explicit_context and len(explicit_context) > 50:  # Substantial explicit context
        entities_with_explicit_context.append((entity, explicit_context, len(explicit_context)))

# Sort by context richness
entities_with_explicit_context.sort(key=lambda x: x[2], reverse=True)

vprint(f"\n📊 Explicit Context Distribution:")
vprint(f"   Entities with explicit context: {len(entities_with_explicit_context)}")
vprint(f"   Average context length: {sum(length for _, _, length in entities_with_explicit_context) / len(entities_with_explicit_context) if entities_with_explicit_context else 0:.1f} chars")
vprint(f"   Total context across all entities: {sum(length for _, _, length in entities_with_explicit_context)} chars")

# Show entities with the richest explicit context
vprint(f"\n🏆 Top Entities by Explicit Context:")
for i, (entity, context, length) in enumerate(entities_with_explicit_context[:3], 1):
    vprint(f"   {i}. {entity['name']}: {length} chars of explicit context")

# Demonstrate explicit context processing with the entity that has the richest context
if entities_with_explicit_context:
    top_entity_data, top_context, context_length = entities_with_explicit_context[0]
    vprint(f"\n🎯 Processing Entity with Richest Explicit Context: {top_entity_data['name']}")
    vprint(f"   Type: {top_entity_data['entity_type']}")
    vprint(f"   Explicit context length: {context_length} characters")
    vprint(f"   Context preview: {top_context[:100]}...")
    
    # Process with explicit context (no external API calls needed)
    vprint(f"\n   📚 Processing with explicit context (no external API calls)...")
    enriched_entity = entity_enricher.enrich_entity(
        name=top_entity_data['name'],
        source_context=top_context,  # Use explicit context instead of description
        aliases=top_entity_data.get('aliases', [])
    )
    
    # Add missing fields
    enriched_entity.entity_type = top_entity_data['entity_type']
    enriched_entity.description = top_entity_data.get('description', '')
    
    print(f"\n ✅ Processed: {len(enriched_entity.entity_context)} chars, confidence: {enriched_entity.confidence_score:.3f}")
    vprint(f"   📝 Context source: {enriched_entity.enrichment_source}")
    vprint(f"   ⚡ Processing speed: Fast (no external API calls)")
    
    # Index with explicit context
    vprint(f"\n   🔍 Indexing with explicit context...")
    index_result = entity_indexer.index_entity(enriched_entity)
    print(f"   ✅ Indexed successfully with explicit context")
    
    # Show explicit context benefits
    vprint(f"\n   📋 Explicit Context Benefits:")
    vprint(f"      Context source: Dataset (explicit)")
    vprint(f"      API calls saved: 1 Wikipedia lookup")
    vprint(f"      Processing time: ~0.1 seconds (vs ~2+ seconds for Wikipedia)")
    vprint(f"      Context quality: Controlled and consistent")
    vprint(f"      Reliability: High (no external dependencies)")
    
    # Show context comparison
    vprint(f"\n   🔍 Context Comparison:")
    vprint(f"      Original description: {top_entity_data.get('description', '')[:80]}...")
    vprint(f"      Explicit context: {top_context[:80]}...")
    vprint(f"      Context enhancement: {len(top_context) - len(top_entity_data.get('description', ''))} chars added")
    
    # Show how explicit context is more informative
    vprint(f"\n   📈 Context Quality Analysis:")
    vprint(f"      Description length: {len(top_entity_data.get('description', ''))} chars")
    vprint(f"      Explicit context length: {len(top_context)} chars")
    vprint(f"      Information density: {len(top_context) / len(top_entity_data.get('description', '')):.1f}x more detailed")

else:
    print(f"\n❌ No entities with explicit context found in dataset")
    vprint(f"   This indicates the dataset doesn't include explicit context fields")
    vprint(f"   In production, explicit context would be provided for better processing")

print(f"\n✅ Explicit context processing demonstration complete!")
vprint(f"   - Demonstrates the efficiency of explicit context processing")
vprint(f"   - Shows real implementation of context-aware entity handling")
vprint(f"   - Provides insight into context optimization strategies")
print("\n✅ Scenario 1 complete — key takeaway: explicit context should be treated as ground truth when available.")


## Lab Step 14 — Scenario 2 — Multilingual Entity Handling

This step exists to stress multilingual edge cases and confirm our pipeline handles non-Latin scripts safely.

**What you’ll do:** Test multilingual names/aliases and observe handling.


**Checkpoint:** You should see multilingual aliases processed correctly.


In [ ]:
# Lab Step 14 — Scenario Walkthroughs: Context + Multilingual Handling
# Purpose (lab-style):
#   - Show how entity context behaves in two realistic scenarios:
#       (1) Aliases / alternate surface forms
#       (2) Multilingual / non-Latin scripts
#   - In non-verbose mode, we still print a compact, inspectable summary.
#   - In verbose mode, we include side-by-side context previews.

import re

print("🧪 Lab Step 14 — Scenario Walkthroughs: Context + Multilingual Handling")
print("=" * 60)

# --- Preconditions ---
for var_name in ["watch_list", "entity_enricher"]:
    if var_name not in globals():
        raise RuntimeError(f"{var_name} not found. Run the earlier steps that initialize it first.")

entities = list(watch_list.get_all_entities())
if not entities:
    raise ValueError("watch_list contains no entities.")

def _is_non_latin(s: str) -> bool:
    if not s:
        return False
    # crude but effective: contains characters outside basic Latin range
    return any(ord(ch) > 0x024F for ch in s)

def _preview(s: str, n: int = 80) -> str:
    s = (s or "").replace("\n", " ").strip()
    if not s:
        return "—"
    return (s[:n] + "…") if len(s) > n else s

def _confidence(enriched_obj):
    # Different versions name this differently; fall back safely
    for attr in ["context_confidence", "confidence", "context_selection_confidence"]:
        v = getattr(enriched_obj, attr, None)
        if isinstance(v, (int, float)):
            return float(v)
    return None

def _context(enriched_obj) -> str:
    return (getattr(enriched_obj, "entity_context", None) or "").strip()

def _desc(entity_obj) -> str:
    # Your entities sometimes store description in metadata rather than attribute
    d = getattr(entity_obj, "description", None)
    if isinstance(d, str) and d.strip():
        return d.strip()
    md = getattr(entity_obj, "metadata", {}) or {}
    for k in ["explicit_context", "description", "context"]:
        v = md.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def _aliases(entity_obj):
    return list(getattr(entity_obj, "aliases", []) or [])

# ----------------------------
# Scenario 1: Alias handling
# ----------------------------
print("\n🧩 Scenario 1: Alias / Alternate Surface Forms")
print("=" * 50)

alias_entity = next((e for e in entities if len(_aliases(e)) >= 2), None) or entities[0]
alias_name = getattr(alias_entity, "name", "unknown")
alias_aliases = _aliases(alias_entity)
alias_desc = _desc(alias_entity)

# Enrich
enriched_alias = entity_enricher.enrich_entity(
    name=alias_name,
    source_context=alias_desc if alias_desc else None,
    aliases=alias_aliases if alias_aliases else None,
)

alias_ctx = _context(enriched_alias)
alias_conf = _confidence(enriched_alias)

# Non-verbose: compact, inspectable summary
print(f"   Selected entity: {alias_name}")
print(f"   Aliases (sample): {alias_aliases[:5] if alias_aliases else '—'}")
print(f"   Description present: {'yes' if bool(alias_desc) else 'no'}")
print(f"   Enriched context: {len(alias_ctx)} chars" + (f", confidence: {alias_conf:.3f}" if alias_conf is not None else ""))

# Verbose: side-by-side previews
vprint("      🔍 Context Comparison:")
vprint(f"         Original description: {_preview(alias_desc)}")
vprint(f"         Enriched context: {_preview(alias_ctx)}")
if alias_desc and alias_ctx:
    vprint(f"         Context delta: {len(alias_ctx) - len(alias_desc)} chars")

# ----------------------------
# Scenario 2: Multilingual entity handling
# ----------------------------
print("\n🌍 Scenario 2: Multilingual Entity Handling")
print("=" * 50)

multi_entity = next(
    (e for e in entities if _is_non_latin(getattr(e, "name", "")) or any(_is_non_latin(a) for a in _aliases(e))),
    None
)

if multi_entity is None:
    print("   ⚠️ No multilingual/non-Latin entity found in this dataset.")
else:
    multi_name = getattr(multi_entity, "name", "unknown")
    multi_aliases = _aliases(multi_entity)
    multi_desc = _desc(multi_entity)

    enriched_multi = entity_enricher.enrich_entity(
        name=multi_name,
        source_context=multi_desc if multi_desc else None,
        aliases=multi_aliases if multi_aliases else None,
    )

    multi_ctx = _context(enriched_multi)
    multi_conf = _confidence(enriched_multi)

    # Non-verbose: compact, inspectable summary (this is the addition you asked for)
    print(f"   Selected entity: {multi_name}")
    print(f"   Aliases (sample): {multi_aliases[:5] if multi_aliases else '—'}")
    print(f"   Description present: {'yes' if bool(multi_desc) else 'no'}")
    print(f"      ✅ Enriched: {len(multi_ctx)} chars" + (f", confidence: {multi_conf:.3f}" if multi_conf is not None else ""))

    # Verbose: safe comparison (no None slicing)
    vprint(f"      🔍 Context Comparison:")
    vprint(f"         Original description: {_preview(multi_desc)}")
    vprint(f"         Enriched context: {_preview(multi_ctx)}")
    if multi_desc and multi_ctx:
        vprint(f"         Context enhancement: {len(multi_ctx) - len(multi_desc)} chars added")
    else:
        vprint("         Context enhancement: — (no original description to compare)")

print("\n✅ Step 14 complete: scenarios demonstrated with compact outputs (and deeper detail under VERBOSE).")


## ✅ Next steps

If you want more narrative context and design rationale, return to the blog post. From here, continue to the next lab notebook in the series (if applicable) or adapt these steps to your own dataset.